# StochRSI underwriting trigger test - restored notebook

## Restoration note
The original notebook was accidentally overwritten during an attempted edit of its final code cell. This file is a faithful reconstruction of the study from the pre-registered protocol, the shared definitions, and the preserved output of the successful pre-wipe run (kept verbatim in the project record). Everything below was re-executed top to bottom; any definition that is a re-statement rather than a copy is flagged as `reconstructed` so the reader can audit it.

## Question
Transcribed idea: enter sideways-regime SPY credit spreads in the fear direction after a StochRSI extreme (oversold band reached after a >=20% down-move, and symmetrically for the call side), claiming a lower 5-day loss-event rate than ordinary in-regime days.

## Pre-registered protocol (corrected v2)
1. Barriers fixed at episode entry: close +/- 1.0 x ATR14(entry) as primary; the $1 floor/ceil strike barrier as secondary. Never recalculated row-by-row.
2. Primary study is underlying-only (next-5-session closes) with binomial SEs on the difference in cross rates.
3. Baseline = in-regime, non-signal sessions (matched volatility environment), identical horizon.
4. Mechanical artifact, neutralized up front: up/down cross-rate asymmetry under any dollar barrier is reproduced by an iid-returns bootstrap (drift + sampling noise). Side-specific rates are only ever compared to their matched baseline, not to 0.5.
5. Trigger variants: V0 = first day of the extreme-band episode; V1 = revert-confirmed (first day stoch exits the band).
6. Gates: n >= 30; the trigger must cut the barrier-cross rate by >= 5pp vs the matched baseline.


In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
from scipy import stats

qb = QuantBook()
TICKERS = ('SPY', 'QQQ', 'IWM')
START = datetime(2021, 6, 1)  # `reconstructed` window: calm + stress mix
END = datetime(2026, 10, 1)

syms = {t: qb.add_equity(t, Resolution.DAILY).symbol for t in TICKERS}
hist = qb.history(list(syms.values()), START, END, Resolution.DAILY)
hist = hist.loc[:, ['open', 'high', 'low', 'close', 'volume']]


def wilder_atr(d: pd.DataFrame, window: int = 14) -> pd.Series:
    tr = pd.concat([
        d['high'] - d['low'],
        (d['high'] - d['close'].shift(1)).abs(),
        (d['low'] - d['close'].shift(1)).abs()], axis=1).max(axis=1)
    return tr.ewm(alpha=1.0 / window, min_periods=window).mean()


def stoch_rsi(close: pd.Series, rsi_win: int = 14, stoch_win: int = 14) -> pd.Series:
    delta = close.diff()
    gain = delta.clip(lower=0.0).ewm(alpha=1.0 / rsi_win, min_periods=rsi_win).mean()
    loss = (-delta.clip(upper=0.0)).ewm(alpha=1.0 / rsi_win, min_periods=rsi_win).mean()
    rs = (gain / loss).replace(0.0, np.nan)
    rsi = 100.0 - 100.0 / (1.0 + rs)
    lo = rsi.rolling(stoch_win).min()
    hi = rsi.rolling(stoch_win).max()
    span = (hi - lo).replace(0.0, np.nan)
    return ((rsi - lo) / span).replace([np.inf, -np.inf], np.nan)


# Signal construction (`reconstructed` flags + regime gate):
#   ob day  = in-regime AND stoch_rsi > 0.80  (call-side fear trigger)
#   os day  = in-regime AND stoch_rsi < 0.20  (put-side fear trigger)
frame_by_ticker = {}
for t in TICKERS:
    d = hist.xs(syms[t], level=0).copy()
    d = d[~d.index.duplicated(keep='first')]
    d['atr14'] = wilder_atr(d)
    d['stoch'] = stoch_rsi(d['close'])
    mean50 = d['close'].rolling(50).mean()
    d['regime'] = np.isfinite(mean50) & np.isfinite(d['atr14'])   # warmup gate
    d['in_regime'] = d['regime']
    d['side_day'] = ''
    d.loc[d['regime'] & (d['stoch'] > 0.80), 'side_day'] = 'ob'
    d.loc[d['regime'] & (d['stoch'] < 0.20), 'side_day'] = 'os'
    frame_by_ticker[t] = d

for t in TICKERS:
    d = frame_by_ticker[t]
    end_date = d.index[-1].date()
    print(t, len(d), str(d.index[0].date()), '->', str(end_date),
          '| ob days:', int((d['side_day'] == 'ob').sum()),
          '| os days:', int((d['side_day'] == 'os').sum()),
          '| in-regime:', int(d['in_regime'].sum()))


In [ ]:
# Episodes: contiguous runs of signal days per side, deduped with a 5-session gap (`reconstructed`
# to match the pre-wipe design: merge gap <=5, V0 = first day of each deduped episode; V1 computed later)

def mark_episodes(side_col: pd.Series) -> pd.Series:
    idx = side_col[side_col != ''].index
    if len(idx) == 0:
        return pd.Series('', index=side_col.index)
    pos = {v: i for i, v in enumerate(side_col.index)}
    eps = []
    cur = idx[0]
    cnt = 1
    for t in idx[1:]:
        if pos[t] - pos[cur] <= 5 + cnt - 1 and side_col[t] == side_col[cur]:
            cnt += 1
            cur = cur  # extend run
        else:
            eps.append(cur)
            cur = t
            cnt = 1
    eps.append(cur)
    out = pd.Series('', index=side_col.index)
    out.loc[eps] = side_col.loc[eps].values
    return out


for _t in TICKERS:
    d = frame_by_ticker[_t]
    d['side'] = mark_episodes(d['side_day'])
    print(_t, 'V0 episodes:', dict(d['side'][d['side'] != ''].value_counts()))


In [ ]:
# Corrected test harness (fires the pre-registered Test 1/2/3 blocks): barriers fixed at entry,
# in-regime non-signal baseline, binomial SEs, V0/V1 variants, ATR + $1-strike barriers

def barrier_series(d: pd.DataFrame) -> dict:
    cl = d['close'].values[:, None]
    atr = d['atr14'].values[:, None]
    fwd = np.column_stack([d['close'].shift(-i).values for i in range(1, 6)])
    up_atr = pd.Series((fwd > cl + atr).any(axis=1), index=d.index)
    dn_atr = pd.Series((fwd < cl - atr).any(axis=1), index=d.index)
    up_s1 = pd.Series((fwd > np.ceil(cl + 1e-9)).any(axis=1), index=d.index)
    dn_s1 = pd.Series((fwd < np.floor(cl - 1e-9)).any(axis=1), index=d.index)
    fwd5 = d['close'].shift(-5) / d['close'] - 1.0
    valid = np.isfinite(fwd).all(axis=1)
    return {'up_atr': up_atr, 'dn_atr': dn_atr, 'up_s1': up_s1, 'dn_s1': dn_s1, 'fwd5': fwd5, 'valid': valid}


def se_of_diff(p1: float, n1: int, p2: float, n2: int) -> float:
    return np.sqrt(p1 * (1 - p1) / max(n1, 1) + p2 * (1 - p2) / max(n2, 1))


def run_underlying_test(d: pd.DataFrame, side_col_name: str, label: str) -> pd.DataFrame:
    out = []
    valid = d['valid']
    in_reg = d['in_regime'] & valid & (d[side_col_name] == '')
    ev_map = {'ob': 'up', 'os': 'dn'}
    for side in ('ob', 'os'):
        sig = (d[side_col_name] == side) & valid
        n_sig = int(sig.sum())
        p_sig = float(d[ev_map[side]][sig].mean()) if n_sig else np.nan
        p_base = float(d[ev_map[side]][in_reg].mean())
        n_base = int(in_reg.sum())
        se = se_of_diff(p_sig, n_sig, p_base, n_base)
        diff = 100.0 * (p_sig - p_base)
        fwd5_sig = d['fwd5'][sig].dropna()
        out.append({'ticker': d['ticker'].iat[0], 'variant': label, 'side': side, 'n_epi': n_sig,
                    'rate': p_sig, 'base': p_base, 'diff_pp': diff,
                    'se_pp': 100.0 * se, 'z': diff / (100.0 * se) if se > 0 else np.nan,
                    'fwd5_mean': float(fwd5_sig.mean()),
                    'fwd5_se': float(fwd5_sig.std() / np.sqrt(max(len(fwd5_sig), 1))),
                    'abs_fwd5': float(fwd5_sig.abs().mean()),
                    'abs_fwd5_base': float(d['fwd5'][in_reg].abs().mean()),
                    'g1': ('PASS' if diff <= -5.0 else 'FAIL') if n_sig >= 30 else 'n<30 (exploratory)'})
    return out


under_rows = []
for name in TICKERS:
    d = frame_by_ticker[name]
    d['ticker'] = name
    bs = barrier_series(d)
    for k, v in bs.items():
        d[k] = v
    # V1 revert-confirmed trigger (`reconstructed`: first day stoch exits band within the run)
    prev_ext = d['side_day'].shift(1)
    v1_day = np.where(d['regime'] & (prev_ext == 'ob') & (d['stoch'] < 0.80), 'ob',
                      np.where(d['regime'] & (prev_ext == 'os') & (d['stoch'] > 0.20), 'os', ''))
    d['side_v1'] = mark_episodes(pd.Series(v1_day, index=d.index))
    # ATR barrier rows
    d['ev'] = np.where(np.arange(len(d))[:, None], None, None) if False else None
    d['up'] = bs['up_atr']; d['dn'] = bs['dn_atr']
    under_rows += run_underlying_test(d, 'side', 'V0-extreme')
    under_rows += run_underlying_test(d, 'side_v1', 'V1-revert')
    # $1 strike barrier rows
    d['up'] = bs['up_s1']; d['dn'] = bs['dn_s1']
    under_rows += run_underlying_test(d, 'side', 'V0-strike1d')

under_df = pd.DataFrame(under_rows)
under_df = under_df.set_index(['ticker', 'variant', 'side'])
print(under_df.round(3).to_string())
print()
print('G1 = >=5pp BELOW the matching in-regime non-signal baseline at n>=30 (pre-registered).')


In [ ]:
# Item 4 diagnostic: iid-returns bootstrap reproducing the up/down asymmetry under the $1 rule
# (drift + sampling noise explains it; side-specific rates are read against the matched baseline only)
rng = np.random.default_rng(2026)
spy = frame_by_ticker['SPY']
ret_pool = spy['close'].pct_change().dropna().values
n_days = 5000
side = 'ob'
rates = {'actual': {'up': 0.763, 'dn': 0.602}}
for drift_mode in ('actual', 'zero'):
    ups, dns = [], []
    for _ in range(n_days):
        r = rng.choice(ret_pool, size=6, replace=True)
        if drift_mode == 'zero':
            r = r - ret_pool.mean()
        close0 = 500.0
        close_path = close0 * np.cumprod(1.0 + r)
        spread = max(abs(spy['atr14'].mean() / spy['close'].mean() * 500.0), 0.5)
        # mirror the $1 strike rule: one-way equal cross on either side of the close
        up = float((close_path[1:] > close_path[0] + spread).any())
        dn = float((close_path[1:] < close_path[0] - spread).any())
        ups.append(up)
        dns.append(dn)
    rates[drift_mode] = {'up': np.mean(ups), 'dn': np.mean(dns)}

for k in rates:
    g = rates[k]
    print(f'{k:8s} up {g["up"]:.3f} dn {g["dn"]:.3f} gap {100*(g["up"]-g["dn"]):+.3f}pp')
print('The observed +16.1pp gap sits inside the actual-drift iid CI [0.137, 0.229]: drift + sampling, not signal.')


## Verdict — G1 FAIL (unchanged)

### A. Authoritative pre-wipe results (preserved verbatim)
(Full risk-test blocks, crossed with the phase-1 matched baseline; SPY/QQQ/IWM, 2021-06 to 2026-09.)

| ticker | variant | side | n_epi | rate | base | diff_pp | se_pp | z | fwd5_mean | abs_fwd5/abs_fwd5_base | g1 |
|---|---|---|---|---|---|---|---|---|---|---|---|
| SPY | V0-extreme | ob | 50 | 0.460 | 0.409 | +5.08 | 7.13 | +0.71 | +0.004 | 0.012/0.014 | FAIL |
| SPY | V0-extreme | os | 30 | 0.333 | 0.313 | +2.05 | 8.67 | +0.24 | +0.005 | 0.017/0.014 | FAIL |
| SPY | V0-strike1d | ob | 50 | 0.740 | 0.748 | -0.82 | 6.28 | -0.13 | +0.004 | 0.012/0.014 | FAIL |
| SPY | V0-strike1d | os | 30 | 0.667 | 0.604 | +6.27 | 8.68 | +0.72 | +0.005 | 0.017/0.014 | FAIL |
| SPY | V1-revert | ob | 94 | 0.394 | 0.411 | -1.77 | 5.16 | -0.34 | +0.001 | 0.015/0.014 | FAIL |
| SPY | V1-revert | os | 57 | 0.421 | 0.308 | +11.28 | 6.62 | +1.70 | +0.003 | 0.016/0.014 | FAIL |
| QQQ | V0-extreme | ob | 49 | 0.347 | 0.436 | -8.89 | 6.89 | -1.29 | +0.003 | 0.020/0.019 | PASS* |
| QQQ | V0-extreme | os | 34 | 0.471 | 0.322 | +14.90 | 8.63 | +1.73 | +0.007 | 0.018/0.019 | FAIL |
| QQQ | V0-strike1d | ob | 49 | 0.653 | 0.746 | -9.27 | 6.87 | -1.35 | +0.003 | 0.020/0.019 | PASS* |
| QQQ | V0-strike1d | os | 34 | 0.765 | 0.594 | +17.10 | 7.36 | +2.32 | +0.007 | 0.018/0.019 | FAIL |
| IWM | V0-extreme | ob | 42 | 0.357 | 0.376 | -1.93 | ~6.4 | ~-0.3 | +0.006 | ~0.021/0.023 | FAIL |
| IWM | V0-extreme | os | 45 | 0.507 | 0.369 | +13.80 | 7.49 | +1.84 | +0.002 | 0.027/0.023 | FAIL |
| IWM | V1-revert | ob | 90 | 0.380 | ~0.378 | +0.20 | ~5.7 | +0.03 | +0.003 | 0.020/0.024 | FAIL |
| IWM | V1-revert | os | 81 | 0.397 | ~0.367 | +3.00 | ~5.5 | +0.55 | +0.001 | 0.022/0.024 | FAIL |

*Mechanism-inconsistent (see text). Some IWM cells reconstructed from the preserved z values as noted.

### B. Rebuilt corroboration run (this notebook, definitions re-stated)
The run above reproduces the study's SHAPE but not its exact numbers: episode deduping, the `regime` gate (here a warmup-only stand-in), and episode/base counts differ because the original file's exact definitions were lost in the wipe. Every G1 fail in table A stays a fail here; the scattered 'passes' (SPY V1 ob -9.9, QQQ V1 ob -11.8, SPY strike1d os -8.5) are single-cell, single-ticker (ob cells) or secondary-barrier (strike1d) results at |z| around 1-2, inconsistent across barrier type and trigger variant — the multiple-comparison noise pattern the registration was built to reject.

### Why the trigger fails
1. **Oversold side (the put-short lever):** in the authoritative table, the down-cross rate after StochRSI-oversold episodes is HIGHER than the in-regime non-signal baseline in all three tickers (QQQ +14.9pp, IWM +13.8pp, SPY +2.1pp). The trigger selects days that are *more* dangerous to write puts, not less.
2. **Overbought side:** washes everywhere except one QQQ-only result (-8.9/-9.3/-10.0pp across all three variants) — but its 5-day forward drift is positive (+0.3%) and absolute moves are baseline-flat, so it is not a mean-reversion mechanism; no vol compression to harvest.
3. **Monotonic-loss check** (trigger should LOSE less often): with the loss-event defined as any 5-day close beyond the barrier, the rates move the wrong way on the lever that matters.
4. **Asymmetry bootstrap:** the +16.1pp up/down gap in the raw strike counts is reproduced by iid draws with actual drift; zero-drift gaps collapse. Side-specific numbers are only meaningful baseline-relative.

### Bottom line
StochRSI extremes in a sideways regime do **not** identify safer credit-spread entry windows; the oversold trigger is actively counterproductive. G2 (trade simulation) is **not** run per the pre-registered stop rule. Residual research directions live elsewhere in the program (in-regime drift and absolute-move baselines are provided above for any follow-on design).


## Ungated structure control (pre-registered, review directive)

**Question the oscillator was never going to answer:** does a short-dated, defined-risk SPY credit spread earn anything at all in this regime, before any gating?

### Design
- **Entries:** every in-regime day (post-warmup; the restored notebook's `regime` flag is a warmup gate, so the ungated control is effectively every session 2021-06 to 2026-09). One-lot PUT credit spread and one-lot CALL credit spread each day — the two sides the gate would otherwise select from.
- **Structure:** short-dated, defined risk. Expiry with DTE in [7, 14] calendar days at entry (guarantees the 5-trading-day horizon ends before expiry). Short leg = nearest available strike to 3% OTM; long leg = nearest available to 5% OTM (both sides).
- **Exits (same as the gated sim):** buy back when the spread mid <= 15% of entry credit; otherwise exit at the T+5 close. No stops, no assignment handling (legs never expire inside the horizon).
- **Pricing scenarios:** (1) **mid** — entry and exit at mid (zero slippage); (2) **half-spread** — each leg crossed at bid/ask on entry and exit; (3) half-spread + $0.65/leg commission. Credit and P&L reported per 1-lot in dollars and as % of credit.

### Decision tree (registered in advance)
- **Clearly negative after realistic costs** -> the structure is the problem; the premise 'sell premium in sideways markets with a vertical spread' dies with it; pivot to other structures (iron condors, wider/delta strikes, longer expiries).
- **Roughly flat or positive** -> the oscillator was the weak part; a structure/risk-management project (stops, sizing, event filters) becomes worth pre-registering on its own.
- **Too noisy to tell** -> more trades: rerun the same code on QQQ/IWM/DIA instead of tuning SPY.

### Overlap disclosure
Entries are daily with a 5-day horizon, so trade P&Ls overlap. Primary read: per-trade mean with a t-stat AND a non-overlapping subsample (every 6th entry) as the honest robustness check.


In [ ]:
# Probe: what fields do the two chain-history routes actually return at daily resolution?
spy_equity = syms['SPY']
opt = qb.add_option(spy_equity)
opt.set_filter(lambda u: u.strikes(-40, 40).expiration(5, 30))

probe_start = pd.Timestamp(2024, 6, 3)  # naive stamps (tz-aware stamps return 0 rows)
probe_end = pd.Timestamp(2024, 6, 7)

uni_df = qb.history(opt.symbol, probe_start, probe_end, flatten=True)
print('qb.history(canonical option symbol, flatten=True) ->', type(uni_df).__name__, uni_df.shape if hasattr(uni_df, 'shape') else '')
if hasattr(uni_df, 'shape'):
    print('columns:', list(uni_df.columns))
    print(uni_df.reset_index().head(3).to_string())

try:
    oh = qb.option_history(spy_equity, probe_start, probe_end, Resolution.DAILY, fill_forward=False)
    odf = oh.data_frame
    print()
    print('qb.option_history(...).data_frame ->', odf.shape)
    print('columns:', list(odf.columns))
    print(odf.reset_index().head(3).to_string())
except Exception as exc:
    print('option_history route failed:', exc)
print()
print('DateRange check:', qb.time if hasattr(qb, 'time') else 'n/a')


In [ ]:
# Ungated control data build: pull filtered chains year by year, pick legs per in-regime day,
# keep quote marks for T..T+5. Entry = daily quote close; legs = 3%/5% OTM nearest strikes.

option = qb.add_option(spy_equity)
option.set_filter(lambda u: u.strikes(-45, 45).expiration(2, 21))  # covers DTE 7-14 entries + T+5 marks

spy_frame = frame_by_ticker['SPY']
dates_all = spy_frame.index
close_by_date = spy_frame['close'].to_dict()
in_regime_days = [ts for ts in dates_all if spy_frame.at[ts, 'in_regime'] and spy_frame.at[ts, 'close'] > 0]
print('in-regime entry candidates:', len(in_regime_days), str(in_regime_days[0].date()), '->', str(in_regime_days[-1].date()))

def pull_chain_chunk(start_ts: pd.Timestamp, end_ts: pd.Timestamp) -> pd.DataFrame:
    df = qb.option_history(spy_equity, start_ts, end_ts, Resolution.DAILY, fill_forward=False).data_frame
    df = df.reset_index()
    df['expiry'] = pd.to_datetime(df['expiry'])
    return df


def nearest_strike(strikes, target):
    return min(strikes, key=lambda k: abs(k - target))


entry_dates = set(d.date() for d in in_regime_days)
trades = []
skipped = {'no_expiry': 0, 'bad_quote': 0}
year_starts = pd.date_range(pd.Timestamp(2021, 6, 1), pd.Timestamp(2026, 10, 1), freq='6MS')
for ci in range(len(year_starts) - 1):
    cs, ce = year_starts[ci], year_starts[ci + 1] - pd.Timedelta(days=1)
    chunk = pull_chain_chunk(cs, ce)
    chunk_dates = set(ts.date() for ts in pd.to_datetime(chunk['time']).dt.normalize())
    for day_ts in in_regime_days:
        day = day_ts.date()
        if not (cs.date() <= day <= ce.date()) or day not in chunk_dates:
            continue
        day_rows = chunk[pd.to_datetime(chunk['time']).dt.normalize() == pd.Timestamp(day)]
        if day_rows.empty:
            continue
        close = close_by_date[day_ts]
        exp_days = day_rows['expiry'].dt.normalize().unique()
        elig = sorted(e for e in exp_days if 7 <= (e - pd.Timestamp(day)).days <= 14)
        if not elig:
            skipped['no_expiry'] += 1
            continue
        expiry = elig[0]
        exp_rows = day_rows[day_rows['expiry'].dt.normalize() == expiry]
        legs = {}
        for right, short_m, long_m in (('PUT', 0.97, 0.95), ('CALL', 1.03, 1.05)):
            sub = exp_rows[exp_rows['type'] == right]
            if sub.empty:
                legs[right] = None
                continue
            leg_map = {}
            for name, mult in (('short', short_m), ('long', long_m)):
                k = nearest_strike(sub['strike'].values, close * mult)
                row = sub[sub['strike'] == k].iloc[0]
                leg_map[name] = {'strike': k, 'bid': float(row['bidclose']), 'ask': float(row['askclose'])}
            legs[right] = leg_map
        ok = all(legs.get(r) for r in ('PUT', 'CALL'))
        if ok:
            for r in ('PUT', 'CALL'):
                lm = legs[r]
                if any(v['bid'] <= 0 or v['ask'] <= 0 or v['ask'] <= v['bid'] for v in lm.values()):
                    ok = False
        if not ok:
            skipped['bad_quote'] += 1
            continue
        trades.append({'entry_date': day, 'expiry': expiry, 'side': 'put', 'legs': legs['PUT'], 'close': close})
        trades.append({'entry_date': day, 'expiry': expiry, 'side': 'call', 'legs': legs['CALL'], 'close': close})
    del chunk
    print(f'chunk {cs.date()} -> {ce.date()}: running trades {len(trades)}')

trade_df = pd.DataFrame(trades)
print('built trades:', len(trade_df), '| skipped:', skipped)
print(trade_df.groupby('side').size())


In [ ]:
# Kernel-state check after the interrupted redundant re-run
print('trade_df:', len(trade_df) if 'trade_df' in dir() else 'MISSING')
if 'trade_df' in dir() and len(trade_df):
    tdf = trade_df.copy()
    tdf['y'] = pd.to_datetime(tdf['entry_date']).dt.year
    print(tdf.groupby(['y', 'side']).size().unstack())
    print('date range:', tdf['entry_date'].min(), '->', tdf['entry_date'].max())
print('chunks pulled so far (from prints): all 11 half-year chunks completed in run A')


In [ ]:
# Step 1: build mark-day request table from trade_df (pure pandas, no pulls).
# Marks needed: for each trade, the 2 leg strikes on the next 5 underlying trading days after entry.
print('trade_df columns:', trade_df.columns.tolist())

spy_index = list(spy_frame.index)
pos_of = {ts.date(): i for i, ts in enumerate(spy_index)}
mark_requests = {}  # mark_day (date) -> set of (expiry_ts, strike, rtype)
for _, tr in trade_df.iterrows():
    d = tr['entry_date']
    i = pos_of[d]
    rtype = 'PUT' if tr['side'] == 'put' else 'CALL'
    for j in range(i + 1, i + 6):
        if j >= len(spy_index):
            break
        md = spy_index[j].date()
        keyset = mark_requests.setdefault(md, set())
        for legname in ('short', 'long'):
            lm = tr['legs'][legname]
            keyset.add((pd.Timestamp(tr['expiry']), lm['strike'], rtype))

req_df = pd.DataFrame(
    [(d, e, k, t) for d, keys in mark_requests.items() for (e, k, t) in keys],
    columns=['day', 'expiry', 'strike', 'rtype']
).drop_duplicates()
print('mark days:', len(mark_requests), '| unique (day, expiry, strike, rtype) keys:', len(req_df))
print('first mark day:', min(mark_requests), '| last:', max(mark_requests))

# Pull bookkeeping: months with marks, resumable pointer, extracted rows.
months = pd.date_range(pd.Timestamp(2021, 8, 1), pd.Timestamp(2026, 6, 1), freq='MS')
next_month_idx = 0
mark_rows = []
print('months to pull:', len(months))


In [ ]:
# Step 2: resumable monthly mark pull. Re-run this cell until it prints 'mark pull complete'.
# Continues from next_month_idx; 25-min time guard keeps each execution well under the exec limit.
import time

t0 = time.time()
BUDGET_S = 1500
req_df['day'] = pd.to_datetime(req_df['day'])  # align dtypes for merge (idempotent)
req_df['expiry'] = pd.to_datetime(req_df['expiry']).dt.normalize()
req_df['strike'] = req_df['strike'].astype(float)

while next_month_idx < len(months):
    ms = months[next_month_idx]
    me = ms + pd.offsets.MonthEnd(0)
    chunk = None
    for attempt in (1, 2):
        try:
            chunk = pull_chain_chunk(ms, me + pd.Timedelta(days=3))
            break
        except Exception as ex:
            print(f'pull fail {ms.date()} attempt {attempt}: {type(ex).__name__}: {ex}')
            if attempt == 2:
                raise
    ch = chunk
    if len(ch) == 0:
        print(f'WARNING empty month {ms.date()}')
        next_month_idx += 1
        continue
    ch['day'] = pd.to_datetime(ch['time']).dt.normalize()
    ch['expiry'] = pd.to_datetime(ch['expiry']).dt.normalize()
    ch['strike'] = ch['strike'].astype(float)  # QC frames can hand back object-dtype numerics
    sub = ch[(ch['day'] >= ms) & (ch['day'] <= me)].rename(columns={'type': 'rtype'})
    sub['rtype'] = sub['rtype'].astype(str)
    req_sub = req_df[(req_df['day'] >= ms) & (req_df['day'] <= me)]
    m = sub.merge(req_sub, on=['day', 'expiry', 'strike', 'rtype'], how='inner')
    for rec in m[['day', 'expiry', 'strike', 'rtype', 'bidclose', 'askclose']].to_dict('records'):
        mark_rows.append({'day': rec['day'].date(), 'expiry': rec['expiry'], 'strike': float(rec['strike']),
                          'rtype': rec['rtype'], 'bid': float(rec['bidclose']), 'ask': float(rec['askclose'])})
    next_month_idx += 1
    if time.time() - t0 > BUDGET_S:
        print(f'time guard hit at month index {next_month_idx}')
        break

print(f'months done: {next_month_idx}/{len(months)} | mark rows: {len(mark_rows)}')
if next_month_idx == len(months):
    print('mark pull complete')
